# Bank Customer Churn Prediction

**Business question:** Which bank customers are most likely to leave, and what drives it?

**Data:** 10,000 bank customers from Kaggle (Churn_Modelling.csv); about 20% left the bank.

**Approach:** Explore churn by customer attributes, encode categorical variables, then compare five classification models (logistic regression, SVM, decision tree, random forest, gradient boosting) using 5-fold cross-validated AUC.

**Summary of results:** _[fill in after running: best model, test AUC, top churn drivers]_

In [ ]:
# Importing the required packages
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier as dtc # tree algorithm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

Churn_df = pd.read_csv('Churn_Modelling.csv')


**Get info on columns in dataset** 

In [ ]:
Churn_df.describe()

**Find shape of dataset**

In [ ]:
Churn_df.shape

**Display first 5 rows of dataset**

In [ ]:
Churn_df.head()

**Display last 5 rows**

In [ ]:
Churn_df.tail()

**Find unique rows in Geography for converting to categorical values later**

In [ ]:
Churn_df.Geography.unique()

**encode Gender rows**

In [ ]:
Churn_df["Gender"] = Churn_df["Gender"].map({"Female": 0, "Male": 1})

**Count of customers exited and stayed**

In [ ]:
plt.figure(figsize=(6,6))
sns.countplot(x=Churn_df['Exited'])
plt.show()

**Display Column names # of nulls and datatype** 

In [ ]:
Churn_df.info()

**Churn by categorical variables (Gender: 0 = Female, 1 = Male)**

In [ ]:
cat_cols = ["Geography", "Gender", "HasCrCard", "IsActiveMember"]

for col in cat_cols:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    sns.countplot(data=Churn_df, x=col, hue="Exited", ax=axes[0])
    axes[0].set_title(f"Customers by {col}")
    axes[0].legend(title="Exited", labels=["Stayed", "Left"])

    (Churn_df.groupby(col)["Exited"].mean() * 100).plot(kind="bar", color="#E0802B", ax=axes[1])
    axes[1].set_title(f"Churn Rate by {col} (%)")
    axes[1].set_ylabel("% who left")
    axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    plt.show()

**Create Box plots of numeric variables in dataset compared to exited**

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

features = ['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary']
for col in features:
    Churn_df.boxplot(column=col, by='Exited', figsize=(6,6))
    plt.title(col)
plt.show()

**Drop Surname and CustomerId, which are identifiers and don't predict churn**

In [ ]:
Churn_df = Churn_df.drop(columns=['Surname', 'CustomerId'], errors='ignore')
Churn_df.head()

**Correlation values**

In [ ]:

corrM = Churn_df.corr()
corrM = corrM.sort_values(by ='Exited',ascending=False)
print(corrM)

**Create Correlation heatmap**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corrmat = Churn_df.corr()
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corrmat,vmax=.8,  square=True);

**Create boxplots of Exited by Age, IsActiveMember, Balance and Credit Score**

In [ ]:
sns.catplot(x="Exited", y="Age",  kind="box",data=Churn_df)
sns.catplot(x="Exited", y="IsActiveMember",  kind="box",data=Churn_df)
sns.catplot(x="Exited", y="Balance",  kind="box",data=Churn_df);
sns.catplot(x="Exited", y="CreditScore",  kind="box",data=Churn_df);

**Show skewness**

In [ ]:
print("Skewness: %f" % Churn_df['Balance'].skew())

**One-hot encode Geography (France is the baseline)**

In [ ]:
if "Geography" in Churn_df.columns:
    Churn_df = pd.get_dummies(Churn_df, columns=["Geography"], drop_first=True, dtype=int)
print(Churn_df.columns.tolist())

In [ ]:
feature_cols = ['CreditScore', 'Geography_Germany', 'Geography_Spain', 'Gender', 'Age',
                'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember',
                'EstimatedSalary']
target = Churn_df['Exited']
X = Churn_df[feature_cols]
y = target
print(X.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=0)

**Model using a decision tree**

In [ ]:
from sklearn import tree
model = dtc(criterion = 'entropy', max_depth = 4)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

**Determine Accuracy of model**

In [ ]:
accuracy_score(y_test, y_pred)

**Text representation of Decision Tree**

In [ ]:
text_representation = tree.export_text(model)
print(text_representation)


**Create Confusion Matrix**

In [ ]:
pd.crosstab(y_test, y_pred, rownames=['True'], colnames=['Predicted'], margins=True)

**Metrics Classification of model**

In [ ]:
import sklearn.metrics as metrics
print(metrics.classification_report(y_test, y_pred))

**Run Random Forest to see if it is more accurate than Decision tree** 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=170)
clf.fit(X_train,y_train);

**Random forest test-set accuracy**

In [ ]:
y_pred=clf.predict(X_test)
Accuracy = metrics.accuracy_score(y_test, y_pred)
print('Accuracy: %.2f' % (Accuracy*100))


**Run Classification report for Random Forest**

In [ ]:
import sklearn.metrics as metrics
print(metrics.classification_report(y_test, y_pred))

**Compare all five models using 5-fold cross-validated AUC on the training set**

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "Decision Tree": DecisionTreeClassifier(criterion="entropy", max_depth=4),
    "Random Forest": RandomForestClassifier(n_estimators=170, random_state=0),
    "Gradient Boosting": GradientBoostingClassifier(random_state=7),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
results = {name: cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc").mean()
           for name, m in models.items()}
pd.Series(results).sort_values(ascending=False).round(3)

**Final model: gradient boosting, the best cross-validated model, evaluated on the held-out test set**

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay, classification_report

best = GradientBoostingClassifier(random_state=7)
best.fit(X_train, y_train)

probs = best.predict_proba(X_test)[:, 1]
print("Gradient Boosting test AUC:", round(roc_auc_score(y_test, probs), 3))
print(classification_report(y_test, best.predict(X_test)))

RocCurveDisplay.from_predictions(y_test, probs)
plt.title("ROC Curve – Gradient Boosting (Test Set)")
plt.show()

**Top churn drivers**

In [ ]:
pd.Series(best.feature_importances_, index=feature_cols).sort_values().plot(kind="barh", figsize=(8,6))
plt.title("Top Churn Drivers – Gradient Boosting")
plt.show()

## Conclusion

_[fill in after running: which model performed best and why, the top churn drivers, and recommended retention actions, e.g. targeting German customers, inactive members, and customers aged 45+]_